In [0]:

from pyspark.sql import functions as F

SILVER = "workspace.codi_silver.deis_egresos_historico"
DIM_CIE10 = "workspace.codi_silver.dim_diagnostico_cie10"
GOLD = "workspace.codi_gold.agg_egresos_diagnostico"

EXPECTED_TOTAL = 4667259

for tabla in [SILVER, DIM_CIE10]:
    if not spark.catalog.tableExists(tabla):
        raise RuntimeError(f"Falta tabla: {tabla}")

df_egresos = spark.table(SILVER)
df_cie10 = spark.table(DIM_CIE10)

total_origen = df_egresos.count()

assert total_origen == EXPECTED_TOTAL, (
    f"Total inesperado: {total_origen:,}"
)

print("Silver:", f"{total_origen:,}", "registros")
print("Dimensión CIE-10:", f"{df_cie10.count():,}")
print("Fuentes disponibles")


In [0]:

def normalizar_codigo(columna):
    return F.upper(
        F.regexp_replace(
            F.trim(F.col(columna)),
            r"[^A-Za-z0-9]",
            ""
        )
    )

# Dimensión reducida con clave normalizada
dim = df_cie10.select(
    F.col("codigo_normalizado").alias("codigo_cie10"),
    F.col("CAPITULO").alias("capitulo"),
    F.col("GLOSA_CAPITULO").alias("glosa_capitulo")
)

# Comprobar que cada código tenga una sola fila
duplicados = (
    dim.groupBy("codigo_cie10")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

assert duplicados == 0, (
    "La dimensión contiene códigos duplicados"
)

# Preparar hechos
hechos = (
    df_egresos
    .select(
        "_source_year",
        "ANO_EGRESO",
        "REGION_RESIDENCIA",
        "GLOSA_REGION_RESIDENCIA",
        "DIAG1"
    )
    .withColumn(
        "codigo_cie10",
        normalizar_codigo("DIAG1")
    )
)

# Asociación sin eliminar registros
df_enriquecido = hechos.join(
    F.broadcast(dim),
    on="codigo_cie10",
    how="left"
)

print("Modelo enriquecido preparado")


In [0]:

df_gold = (
    df_enriquecido
    .groupBy(
        "_source_year",
        "REGION_RESIDENCIA",
        "GLOSA_REGION_RESIDENCIA",
        "capitulo",
        "glosa_capitulo"
    )
    .agg(
        F.count("*").alias("total_egresos"),

        F.sum(
            F.when(
                F.col("ANO_EGRESO") == "*", 1
            ).otherwise(0)
        ).alias("egresos_anio_especial"),

        F.sum(
            F.when(
                F.col("capitulo").isNull(), 1
            ).otherwise(0)
        ).alias("sin_equivalencia_cie10")
    )
    .withColumnRenamed(
        "_source_year", "anio_archivo"
    )
    .withColumn(
        "fecha_procesamiento",
        F.current_timestamp()
    )
)

display(
    df_gold.orderBy(
        "anio_archivo",
        F.desc("total_egresos")
    ).limit(20)
)


In [0]:

# Conteo después del JOIN
total_enriquecido = df_enriquecido.count()

# Suma de registros agregados
resumen = df_gold.agg(
    F.sum("total_egresos").alias("total_gold"),
    F.sum("sin_equivalencia_cie10").alias("sin_cie10"),
    F.sum("egresos_anio_especial").alias("asteriscos")
).first()

total_gold = resumen["total_gold"]

print("=" * 55)
print("CONCILIACIÓN SILVER -> GOLD")
print("=" * 55)

print(f"Silver:        {total_origen:,}")
print(f"Enriquecido:   {total_enriquecido:,}")
print(f"Gold:          {total_gold:,}")
print(f"Diferencia:    {total_origen - total_gold:,}")
print(f"Sin CIE-10:    {resumen['sin_cie10']:,}")
print(f"Año especial:  {resumen['asteriscos']:,}")

assert total_enriquecido == total_origen, (
    "El JOIN modificó la cantidad de registros"
)

assert total_gold == total_origen, (
    "La agregación Gold presenta diferencias"
)

assert resumen["asteriscos"] == 130539, (
    "Descuadre en conservación de asteriscos"
)

print("CONCILIACIÓN APROBADA")


In [0]:

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.codi_gold"
)

if spark.catalog.tableExists(GOLD):
    print("La tabla Gold ya existe.")
    print("No se sobrescribió.")
else:
    (
        df_gold.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD)
    )
    print("Tabla Gold creada:", GOLD)

df_gold_check = spark.table(GOLD)

total_persistido = (
    df_gold_check
    .agg(F.sum("total_egresos").alias("total"))
    .first()["total"]
)

assert total_persistido == EXPECTED_TOTAL, (
    "La tabla Gold persistida no concilia"
)

print("Validación Gold: OK")
print("Egresos representados:", f"{total_persistido:,}")


In [0]:

%sql

SELECT
    anio_archivo,
    capitulo,
    glosa_capitulo,
    SUM(total_egresos) AS egresos
FROM workspace.codi_gold.agg_egresos_diagnostico
GROUP BY
    anio_archivo,
    capitulo,
    glosa_capitulo
ORDER BY
    anio_archivo,
    egresos DESC;
